In [12]:
import pandas as pd 
import numpy as np

 ## 1. LOAD AND INSPECT DATA

In [19]:
customers = pd.read_csv("C:\\Users\\Tushar\\data analyst1\\MamaEarth.Project\\Data\\customers.csv")
orders=pd.read_csv("C:\\Users\\Tushar\\data analyst1\\MamaEarth.Project\\Data\\orders.csv")
products=pd.read_csv("C:\\Users\\Tushar\\data analyst1\\MamaEarth.Project\\Data\\products.csv")

print("Customers shape:", customers.shape)
print("Products shape:", products.shape)

print("Orders shape:", orders.shape)
print("\nOrders information:")
print(orders.info())

print("\nMissing values before cleaning:")
print(orders.isnull().sum())

print("\nRaw payment method values:")
print(orders["payment_method"].value_counts(dropna=False))





Customers shape: (45, 6)
Products shape: (16, 4)
Orders shape: (180, 9)

Orders information:
<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        180 non-null    int64  
 1   customer_id     180 non-null    int64  
 2   product_id      180 non-null    int64  
 3   order_date      180 non-null    str    
 4   quantity        180 non-null    int64  
 5   discount_pct    160 non-null    float64
 6   payment_method  180 non-null    str    
 7   rating          160 non-null    float64
 8   is_returned     180 non-null    int64  
dtypes: float64(2), int64(5), str(2)
memory usage: 15.2 KB
None

Missing values before cleaning:
order_id           0
customer_id        0
product_id         0
order_date         0
quantity           0
discount_pct      20
payment_method     0
rating            20
is_returned        0
dtype: int64

Raw payment method 

## 2. STANDARDISE PAYMENT METHODS

In [22]:
orders["payment_method"] = (
    orders["payment_method"]
    .astype("string")
    .str.strip()
    .str.upper()
)
print(orders["payment_method"] )
print(orders["payment_method"].value_counts(dropna=False))


0       UPI
1       COD
2       UPI
3       COD
4       COD
       ... 
175     COD
176     COD
177     UPI
178    CARD
179     COD
Name: payment_method, Length: 180, dtype: string
payment_method
COD     75
UPI     65
CARD    40
Name: count, dtype: int64[pyarrow]


## 3. DETECT AND REMOVE DOUBLE-SUBMITTED ORDERS

In [25]:
natural_key = [
    "customer_id",
    "product_id",
    "order_date",
    "quantity"
]

duplicate_mask = orders.duplicated(
    subset=natural_key,
    keep="first"
)

duplicate_rows = orders.loc[duplicate_mask].copy()

print("\n========== 3. DUPLICATE CHECK ==========")
print("Potential duplicate rows:", len(duplicate_rows))


if not duplicate_rows.empty:
    print("\nPotential duplicate order IDs:")
    print(duplicate_rows["order_id"].tolist())

    print("\nRows requiring duplicate review:")
    print(
        orders.loc[
            orders.duplicated(
                subset=natural_key,
                keep=False
            )
        ].sort_values(natural_key).to_string(index=False)
    )


orders_clean = orders.drop_duplicates(
    subset=natural_key,
    keep="first"
).copy()

print("\nOrders before duplicate removal:", len(orders))
print("Orders after duplicate removal:", len(orders_clean))
print("Rows removed:", len(orders) - len(orders_clean))



========== 3. DUPLICATE CHECK ==========
Potential duplicate rows: 5

Potential duplicate order IDs:
[1091, 1109, 1126, 1145, 1165]

Rows requiring duplicate review:
 order_id  customer_id  product_id order_date  quantity  discount_pct payment_method  rating  is_returned
     1090          120         210 2026-02-19         3          20.0           CARD     NaN            0
     1091          120         210 2026-02-19         3          20.0           CARD     NaN            0
     1108          121         205 2026-04-04         1          20.0            UPI     4.0            0
     1109          121         205 2026-04-04         1          20.0            UPI     4.0            0
     1164          122         201 2026-08-08         3           5.0            COD     5.0            0
     1165          122         201 2026-08-08         3           5.0            COD     5.0            0
     1144          128         210 2026-06-29         1          15.0            COD     5.

# --------------------------------------------------
# 4. HANDLE MISSING VALUES AND DATA TYPES
# --------------------------------------------------

In [27]:
orders_clean["discount_pct"] = pd.to_numeric(
    orders_clean["discount_pct"],
    errors="coerce"
)

orders_clean["rating"] = pd.to_numeric(
    orders_clean["rating"],
    errors="coerce"
)

orders_clean["quantity"] = pd.to_numeric(
    orders_clean["quantity"],
    errors="coerce"
)

orders_clean["discount_pct"] = (
    orders_clean["discount_pct"].fillna(0) )

rating_median = orders_clean["rating"].median()

orders_clean["rating"] = (
    orders_clean["rating"].fillna(rating_median)
)

orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print("\n========== 4. MISSING VALUE HANDLING ==========")
print("Median rating used:", rating_median)
print("\nMissing values after cleaning:")
print(orders_clean.isnull().sum())


========== 4. MISSING VALUE HANDLING ==========
Median rating used: 4.0

Missing values after cleaning:
order_id          0
customer_id       0
product_id        0
order_date        0
quantity          0
discount_pct      0
payment_method    0
rating            0
is_returned       0
dtype: int64


# --------------------------------------------------
# 5. MERGE TABLES AND RECONCILE ORDER VALUE
# --------------------------------------------------

In [29]:
products["price"] = pd.to_numeric(
    products["price"],
    errors="coerce"
)

customers_for_merge = customers.copy()
products_for_merge = products.copy()

raw_orders_for_revenue = orders.copy()

raw_orders_for_revenue["discount_pct"] = pd.to_numeric(
    raw_orders_for_revenue["discount_pct"],
    errors="coerce"
).fillna(0)

raw_merged = raw_orders_for_revenue.merge(
    products_for_merge[["product_id", "price"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

raw_merged["order_value"] = (
    raw_merged["price"]
    * pd.to_numeric(raw_merged["quantity"], errors="coerce")
    * (1 - raw_merged["discount_pct"] / 100)
)

raw_revenue = raw_merged["order_value"].sum()

# Merge cleaned orders with products and customers.
df = orders_clean.merge(
    products_for_merge,
    on="product_id",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    customers_for_merge,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Revenue after discount; returned orders are still
# included in this measure.
df["order_value"] = (
    df["price"]
    * df["quantity"]
    * (1 - df["discount_pct"] / 100)
)

# Additional business measure excluding returned orders.
# This is retained order value, not verified accounting revenue.
df["retained_value"] = np.where(
    df["is_returned"].astype(str).str.strip().str.lower()
    .isin(["1", "yes", "true", "returned"]),
    0,
    df["order_value"]
)

print("\n========== 5. REVENUE RECONCILIATION ==========")
print(f"Raw discounted order value:     ₹{raw_revenue:,.2f}")
print(
    "Cleaned discounted order value: "
    f"₹{df['order_value'].sum():,.2f}"
)
print(
    "Difference after cleaning:     "
    f"₹{raw_revenue - df['order_value'].sum():,.2f}"
)
print(
    "Retained order value:           "
    f"₹{df['retained_value'].sum():,.2f}"
)

print(
    "\nExplanation: the raw total can differ from the "
    "cleaned total because double-submitted orders are "
    "removed. Missing discounts are treated as 0%, so "
    "imputing those discounts alone should not change "
    "their calculated order values."
)

print("\nRows with missing product price:", df["price"].isna().sum())
print("Rows with missing customer city:", df["city"].isna().sum())


========== 5. REVENUE RECONCILIATION ==========
Raw discounted order value:     ₹168,145.50
Cleaned discounted order value: ₹164,866.75
Difference after cleaning:     ₹3,278.75
Retained order value:           ₹138,295.45

Explanation: the raw total can differ from the cleaned total because double-submitted orders are removed. Missing discounts are treated as 0%, so imputing those discounts alone should not change their calculated order values.

Rows with missing product price: 0
Rows with missing customer city: 0


# --------------------------------------------------
# 6. IQR OUTLIERS: QUANTITY
# --------------------------------------------------

In [30]:
q1 = df["quantity"].quantile(0.25)
q3 = df["quantity"].quantile(0.75)

iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df["is_outlier"] = (
    (df["quantity"] < lower_bound)
    | (df["quantity"] > upper_bound)
)

outliers = df.loc[
    df["is_outlier"],
    ["order_id", "quantity", "order_value"]
]

print("\n========== 6. QUANTITY OUTLIERS ==========")
print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")
print("Outlier count:", len(outliers))
print("\nFlagged outlier orders:")
print(outliers.to_string(index=False))


========== 6. QUANTITY OUTLIERS ==========
Q1: 1.0
Q3: 3.0
IQR: 2.0
Lower bound: -2.0
Upper bound: 6.0
Outlier count: 2

Flagged outlier orders:
 order_id  quantity  order_value
     1143        40      11168.0
     1148        25      10980.0


# --------------------------------------------------
# 7. TEST COD RETURN HYPOTHESIS
# --------------------------------------------------

In [31]:
# H0: COD orders do not have a higher return rate
#     than non-COD orders.
# H1: COD orders have a higher return rate.
#
# This is a descriptive comparison, not proof of causation.

def is_returned(value):
    return str(value).strip().lower() in {
        "1", "yes", "true", "returned"
    }

df["returned_flag"] = df["is_returned"].apply(is_returned)

payment_summary = (
    df.groupby("payment_method", dropna=False)
    .agg(
        total_orders=("order_id", "count"),
        returned_orders=("returned_flag", "sum")
    )
    .reset_index()
)

payment_summary["return_rate_pct"] = (
    payment_summary["returned_orders"]
    / payment_summary["total_orders"]
    * 100
).round(2)

print("\n========== 7. RETURN RATE BY PAYMENT ==========")
print(payment_summary.to_string(index=False))

cod = df["payment_method"].eq("COD")
cod_rate = df.loc[cod, "returned_flag"].mean() * 100
non_cod_rate = df.loc[~cod, "returned_flag"].mean() * 100

if cod.sum() == 0 or (~cod).sum() == 0:
    verdict = "Insufficient data to compare COD and non-COD."
elif cod_rate > non_cod_rate:
    verdict = "SUPPORTED descriptively: COD return rate is higher."
elif cod_rate < non_cod_rate:
    verdict = "NOT SUPPORTED descriptively: COD return rate is lower."
else:
    verdict = "INCONCLUSIVE descriptively: return rates are equal."

print(f"\nCOD return rate: {cod_rate:.2f}%")
print(f"Non-COD return rate: {non_cod_rate:.2f}%")
print("Hypothesis verdict:", verdict)


========== 7. RETURN RATE BY PAYMENT ==========
payment_method  total_orders  returned_orders  return_rate_pct
          CARD            39                8            20.51
           COD            73               11            15.07
           UPI            63               13            20.63

COD return rate: 15.07%
Non-COD return rate: 20.59%
Hypothesis verdict: NOT SUPPORTED descriptively: COD return rate is lower.


# --------------------------------------------------
# 8. PAYMENT METHOD × CITY TIER SEGMENTATION
# --------------------------------------------------

In [33]:
segment_summary = (
    df.groupby(
        ["payment_method", "city_tier"],
        dropna=False
    )
    .agg(
        total_orders=("order_id", "count"),
        returned_orders=("returned_flag", "sum")
    )
    .reset_index()
)

segment_summary["return_rate_pct"] = (
    segment_summary["returned_orders"]
    / segment_summary["total_orders"]
    * 100
).round(2)

segment_summary = segment_summary.sort_values(
    ["return_rate_pct", "total_orders"],
    ascending=[False, False]
)

print("\n========== 8. PAYMENT × CITY TIER ==========")
print(segment_summary.to_string(index=False))

if not segment_summary.empty:
    highest_risk = segment_summary.iloc[0]
    print("\nHighest return-rate segment:")
    print(
        f"Payment: {highest_risk['payment_method']}, "
        f"City tier: {highest_risk['city_tier']}, "
        f"Return rate: {highest_risk['return_rate_pct']:.2f}%, "
        f"Orders: {highest_risk['total_orders']}"
    )


========== 8. PAYMENT × CITY TIER ==========
payment_method  city_tier  total_orders  returned_orders  return_rate_pct
          CARD          2             9                3            33.33
           COD          3            20                5            25.00
           UPI          2            21                5            23.81
           UPI          3            21                5            23.81
          CARD          1            16                3            18.75
           COD          1            35                6            17.14
           UPI          1            21                3            14.29
          CARD          3            14                2            14.29
           COD          2            18                0             0.00

Highest return-rate segment:
Payment: CARD, City tier: 2, Return rate: 33.33%, Orders: 9


# --------------------------------------------------
# 9. CORRELATION ANALYSIS
# --------------------------------------------------


In [34]:
correlation_columns = [
    "quantity",
    "discount_pct",
    "rating",
    "returned_flag",
    "order_value"
]

correlation_data = df[correlation_columns].copy()

correlation_matrix = correlation_data.corr(
    numeric_only=True
)

print("\n========== 9. CORRELATION MATRIX ==========")
print(correlation_matrix.round(3).to_string())


def strength_label(correlation):
    value = abs(correlation)

    if value < 0.10:
        return "Negligible"
    elif value < 0.30:
        return "Weak"
    elif value < 0.50:
        return "Moderate"
    return "Strong"


print("\nCorrelation strength labels:")

columns = correlation_matrix.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        correlation = correlation_matrix.iloc[i, j]

        if pd.notna(correlation):
            print(
                f"{columns[i]} vs {columns[j]}: "
                f"{correlation:.3f} "
                f"({strength_label(correlation)})"
            )



========== 9. CORRELATION MATRIX ==========
               quantity  discount_pct  rating  returned_flag  order_value
quantity          1.000         0.091   0.051         -0.029        0.954
discount_pct      0.091         1.000  -0.035          0.062        0.023
rating            0.051        -0.035   1.000         -0.497        0.044
returned_flag    -0.029         0.062  -0.497          1.000       -0.045
order_value       0.954         0.023   0.044         -0.045        1.000

Correlation strength labels:
quantity vs discount_pct: 0.091 (Negligible)
quantity vs rating: 0.051 (Negligible)
quantity vs returned_flag: -0.029 (Negligible)
quantity vs order_value: 0.954 (Strong)
discount_pct vs rating: -0.035 (Negligible)
discount_pct vs returned_flag: 0.062 (Negligible)
discount_pct vs order_value: 0.023 (Negligible)
rating vs returned_flag: -0.497 (Moderate)
rating vs order_value: 0.044 (Negligible)
returned_flag vs order_value: -0.045 (Negligible)


# --------------------------------------------------
# 10. MONTHLY REVENUE WITH AND WITHOUT OUTLIERS
# --------------------------------------------------


In [36]:
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

monthly = (
    df.groupby("order_month")
    .agg(
        revenue_with_outliers=("order_value", "sum"),
        revenue_without_outliers=(
            "order_value",
            lambda s: s.loc[~df.loc[s.index, "is_outlier"]].sum()
        ),
        retained_value=("retained_value", "sum"),
        order_count=("order_id", "count")
    )
    .reset_index()
    .sort_values("order_month")
)

print("\n========== 10. MONTHLY REVENUE ==========")
print(monthly.round(2).to_string(index=False))

if not monthly.empty:
    peak_row = monthly.loc[
        monthly["revenue_without_outliers"].idxmax()
    ]

    print(
        "\nPeak month after excluding quantity outliers: "
        f"{peak_row['order_month']}"
    )
    print(
        "Corrected monthly revenue: "
        f"₹{peak_row['revenue_without_outliers']:,.2f}")


========== 10. MONTHLY REVENUE ==========
order_month  revenue_with_outliers  revenue_without_outliers  retained_value  order_count
    2025-04                7814.80                   7814.80         6717.85           10
    2025-05                5686.40                   5686.40         2209.10            7
    2025-06                4051.40                   4051.40         4051.40            6
    2025-07               10393.65                  10393.65         4983.90           10
    2025-08                6799.55                   6799.55         4914.05           10
    2025-09                9028.25                   9028.25         7841.65           11
    2025-10                4029.55                   4029.55         4029.55            6
    2025-11                7285.20                   7285.20         6387.00            9
    2025-12                4230.60                   4230.60         3154.20            8
    2026-01                4887.40                   4887